# Introduction

The goal is to verify the alignment of the sample peaks in a batch and make sure batch aggregated peaks are correct.

# Database Initialization

In [ ]:
from mascope_backend.db import init_db

await init_db()

# Data Collection

The pipeline for data collection is taken from `get_sample_batch_peaks` controller. However, the pipeline will be simplified here for clarity:
- intensity_variable is hardcoded
- only one ionization mode is picked (the most frequently occurring one)

In [ ]:
import numpy as np
from mascope_backend.api.controllers.sample.batches.sample_batches_controller import (
    _collect_spectra_per_ionization_mode,
    get_sample_batch_peaks,
)

sample_batch_id = "EHcWyasdCruYN9JT"

spectra, _ = await _collect_spectra_per_ionization_mode(
    sample_batch_id
)
batch_peaks = (
    await get_sample_batch_peaks(
        sample_batch_id,
    )
)["data"]

# Concatenate all spectra for each ionization mode
all_mzs = np.array([])
all_intensities = np.array([])
all_resolutions = np.array([])
for ion_mode, specs in spectra.items():
    for spec in specs:
        all_mzs = np.concatenate([all_mzs, spec.mz])
        all_intensities = np.concatenate([all_intensities, spec.intensity])
        all_resolutions = np.concatenate([all_resolutions, spec.resolution])

# Sort by mz
sort_indices = np.argsort(all_mzs)
all_mzs = all_mzs[sort_indices]
all_intensities = all_intensities[sort_indices]
all_resolutions = all_resolutions[sort_indices]


# Visualization

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

current_idx = 0

def plot_peaks(idx):
    clear_output(wait=True)
    mz = batch_peaks["mzs"][idx]
    # Interpolate resolution at the batch peak m/z
    resolution = np.interp(mz, all_mzs, all_resolutions)
    fwhm = mz / resolution
    x_range = mz-fwhm, mz+fwhm
    mz_mask = (all_mzs >= x_range[0]) & (all_mzs <= x_range[1])

    plt.figure(figsize=(10, 6))
    plt.axvline(x=mz, color="#882020", linestyle='-', linewidth=3, label='Batch Peak')
    plt.axvspan(mz - fwhm / 2, mz + fwhm / 2, color='orange', alpha=0.2, label='FWHM Band')
    plt.scatter(x=all_mzs[mz_mask], y=all_intensities[mz_mask], s=100, alpha=0.5, label='Non-Aligned Sample Peaks')
    plt.title(f'Batch Peak {idx+1}/{len(batch_peaks["mzs"])}: {mz:.4f} m/z')
    plt.xlabel('m/z')
    plt.ylabel('Intensity')
    plt.xlim(x_range)
    plt.legend()
    plt.show()
    display(widgets.HBox([prev_button, next_button]))

def on_next(b):
    global current_idx
    if current_idx < len(batch_peaks["mzs"]) - 1:
        current_idx += 1
    plot_peaks(current_idx)

def on_prev(b):
    global current_idx
    if current_idx > 0:
        current_idx -= 1
    plot_peaks(current_idx)

prev_button = widgets.Button(description="Previous")
next_button = widgets.Button(description="Next")
prev_button.on_click(on_prev)
next_button.on_click(on_next)

plot_peaks(current_idx)